# 🇹🇷 Turkspell: Google Colab Ücretsiz GPU (0 TL / Sıfır Kredi) ile Derlem Madenciliği

Bu notebook, **hiçbir ücret ödemeden ve ücretli kredi (CCU) harcamadan**, tamamen Google Colab'ın **Ücretsiz T4 GPU** katmanı üzerinde Türkçe derlem madenciliği yapmak üzere tasarlanmıştır.

### 🌟 Temel Özellikler:
1. **%100 Ücretsiz ($0 Cost)**: Colab Free Tier donanım limitleri (T4 15 GB VRAM) için 4-bit optimize edilmiştir.
2. **TS Corpus (LexiTR) Entegrasyonu**: Bulunan kelimeler [TS Corpus](https://tscorpus.com/corpora) API'si üzerinden taranarak akademik ve bilgilendirici metin sıklıklarına göre puanlanır.
3. **Kesintisiz Google Drive Yedeklemesi**: Her 50 kelimede bir Google Drive'a kaydedilir, oturum kopsa dahi kaldığı yerden devam eder.
4. **4-Bit Qwen2.5-7B Morfoloji Motoru**: ~5.5 GB VRAM kullanarak yüksek hızda kök, POS ve ek analizi yapar.

---

## ⚡ 1. Adım: GPU Kontrolü ve Google Drive Bağlantısı

Menüden **Çalışma Zamanı (Runtime) -> Çalışma zamanı türünü değiştir** seçeneğine gidin.
- Donanım hızlandırıcı: **T4 GPU (Ücretsiz)** seçin.

In [ ]:
!nvidia-smi

import os
from google.colab import drive

# Google Drive'ı bağla (sonuçları ve ara kontrol noktalarını kaybetmemek için)
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/turkspell_mining'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Google Drive çıktı klasörü hazır: {DRIVE_DIR}")

## 📦 2. Adım: Hunspell Linux ve Python Kütüphanelerinin Kurulumu

In [ ]:
!apt-get update -qq
!apt-get install -y -qq hunspell libhunspell-dev
!pip install -q transformers bitsandbytes accelerate datasets tqdm
print("Tüm bağımlılıklar başarıyla kuruldu!")

## 📂 3. Adım: Turkspell Deposunun Hazırlanması

In [ ]:
import subprocess
import os

# Turkspell deposunu klonla veya en güncel haline çek
if not os.path.exists('/content/turkspell'):
    !git clone https://github.com/selimsum/turkspell.git /content/turkspell
else:
    %cd /content/turkspell
    !git pull origin master

%cd /content/turkspell

# Hunspell'in çalıştığını doğrula
res = subprocess.run(['hunspell', '-d', 'tr', '-l'], input='merhaba\nxyzabc\n', text=True, capture_output=True)
print("Hunspell test sonucu (hatalı kelime çıktısı):")
print(res.stdout)

## 🌐 4. Adım: Derlem Akışı (Streaming) ve Hunspell Taraması

Bu adımda Hugging Face üzerindeki Türkçe Wikipedia makaleleri sabit diske indirilmeden canlı olarak (stream) taranır. Hunspell tarafından tanınmayan kelimeler sıklık sırasına göre toplanır.

In [ ]:
import re
import json
from collections import Counter
from datasets import load_dataset
from tqdm.auto import tqdm

CHECKPOINT_CANDIDATES = os.path.join(DRIVE_DIR, 'mined_candidates_raw.json')

word_freq = Counter()
if os.path.exists(CHECKPOINT_CANDIDATES):
    with open(CHECKPOINT_CANDIDATES, 'r', encoding='utf-8') as f:
        word_freq.update(json.load(f))
    print(f"Drive'daki mevcut aday listesi yüklendi: {len(word_freq):,} kelime.")

print("Türkçe Wikipedia derlem akışı başlatılıyor...")
dataset = load_dataset("wikimedia/wikipedia", "20231101.tr", split="train", streaming=True)

def turkish_lower(s):
    return s.replace('İ', 'i').replace('I', 'ı').lower()

MAX_ARTICLES = 25000
BATCH_TOKENS = 3000
token_buffer = []

article_count = 0
for article in tqdm(dataset, total=MAX_ARTICLES, desc="Makale Tarama"):
    text = article.get('text', '')
    tokens = re.findall(r'\b[a-zA-ZçğıöşüÇĞİÖŞÜâîûÂÎÛ]{3,35}\b', text)
    for t in tokens:
        lt = turkish_lower(t)
        if not lt.isupper() and not any(ch.isdigit() for ch in lt):
            token_buffer.append(lt)

    if len(token_buffer) >= BATCH_TOKENS:
        unique_batch = list(set(token_buffer))
        p = subprocess.run(['hunspell', '-d', 'tr', '-l'], input='\n'.join(unique_batch) + '\n', text=True, capture_output=True)
        flagged = set(p.stdout.splitlines())
        for w in token_buffer:
            if w in flagged:
                word_freq[w] += 1
        token_buffer.clear()

    article_count += 1
    if article_count >= MAX_ARTICLES:
        break

# Kalan tamponu da Hunspell ile tara
if token_buffer:
    unique_batch = list(set(token_buffer))
    p = subprocess.run(['hunspell', '-d', 'tr', '-l'], input='\n'.join(unique_batch) + '\n', text=True, capture_output=True)
    flagged = set(p.stdout.splitlines())
    for w in token_buffer:
        if w in flagged:
            word_freq[w] += 1
    token_buffer.clear()

# En çok geçen ilk 15.000 adayı Drive'a kaydet
top_candidates = dict(word_freq.most_common(15000))
with open(CHECKPOINT_CANDIDATES, 'w', encoding='utf-8') as f:
    json.dump(top_candidates, f, ensure_ascii=False, indent=2)

print(f"\nTarama tamamlandı! Toplam {len(top_candidates):,} aday kelime Drive'a kaydedildi.")
print(f"En sık geçen ilk 10 aday: {list(top_candidates.items())[:10]}")

## 🔍 5. Adım: TS Corpus (LexiTR API) ile Akademik & Bilgilendirici Prestij Zenginleştirmesi

Bu adımda bulunan kelimelerin [TS Corpus](https://tscorpus.com) üzerindeki tür bazlı sıklıkları (Akademik, Bilgilendirici, Kurgusal, Sosyal Medya) sorgulanır. Kelimeler prestij puanına ve derlem sıklığına göre sıralanır.

In [ ]:
import sys
from pathlib import Path
sys.path.append('/content/turkspell')
from tools.ts_corpus_validator import batch_validate_candidates

# Eğer 4. adım önceden çalıştırıldıysa adayları yükle
if 'top_candidates' not in globals() or not top_candidates:
    if os.path.exists(CHECKPOINT_CANDIDATES):
        with open(CHECKPOINT_CANDIDATES, 'r', encoding='utf-8') as f:
            top_candidates = json.load(f)
    else:
        raise ValueError("Aday listesi boş! Lütfen önce 4. adımı çalıştırarak kelime toplayın.")

TS_CACHE_PATH = os.path.join(DRIVE_DIR, 'lexitr_cache.json')
candidate_words = list(top_candidates.keys())[:3000] # En yüksek sıklıklı ilk 3.000 aday

print(f"TS Corpus üzerinden {len(candidate_words)} kelime doğrulanıyor...")
enriched_metrics = batch_validate_candidates(
    candidate_words,
    max_workers=6,
    cache_path=Path(TS_CACHE_PATH),
    save_interval=100
)

# Adayları prestij puanı ve derlem sıklığı ile birleştir
filtered_candidates = {}
for w in candidate_words:
    metrics = enriched_metrics.get(w, {})
    corpus_count = top_candidates.get(w, 1)
    prestige = metrics.get('prestige_score', 0.0)
    filtered_candidates[w] = {
        "corpus_count": corpus_count,
        "prestige_score": prestige,
        "academic_freq": metrics.get("academic_freq", 0),
        "informative_freq": metrics.get("informative_freq", 0),
        "social_freq": metrics.get("social_freq", 0),
        "total_freq": metrics.get("total_freq", 0),
        "is_prestigious": metrics.get("is_prestigious", False)
    }

# Sıralama: Prestij skoru yüksek olanlar ve derlemde en çok geçenler en başta
filtered_candidates = dict(
    sorted(filtered_candidates.items(), key=lambda kv: (kv[1]["prestige_score"], kv[1]["corpus_count"]), reverse=True)
)

FILTERED_CANDIDATES_PATH = os.path.join(DRIVE_DIR, 'prestigious_candidates.json')
with open(FILTERED_CANDIDATES_PATH, 'w', encoding='utf-8') as f:
    json.dump(filtered_candidates, f, ensure_ascii=False, indent=2)

print(f"Zenginleştirme tamamlandı! Toplam {len(filtered_candidates):,} aday analize hazırlandı.")
print(f"İlk 5 aday: {list(filtered_candidates.keys())[:5]}")

## 🤖 6. Adım: 4-Bit Qwen2.5-7B ile Ücretsiz T4 GPU Morfolojik Analizi

Model 4-bit (NF4) olarak yüklenir (~5.5 GB VRAM kaplar).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
print(f"4-Bit model yükleniyor: {MODEL_ID}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

print(f"Model hazır! Ayrılan VRAM: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")

## ⚡ 7. Adım: Toplu Çıkarım (Batch Inference) ve Kesintisiz Checkpointing

In [ ]:
CHECKPOINT_PARSED = os.path.join(DRIVE_DIR, 'parsed_results_checkpoint.json')
parsed_results = {}
if os.path.exists(CHECKPOINT_PARSED):
    try:
        with open(CHECKPOINT_PARSED, 'r', encoding='utf-8') as f:
            parsed_results = json.load(f)
        print(f"Mevcut çözümlenmiş kelimeler yüklendi: {len(parsed_results):,}")
    except Exception:
        parsed_results = {}

if 'filtered_candidates' not in globals() or not filtered_candidates:
    if os.path.exists(FILTERED_CANDIDATES_PATH):
        with open(FILTERED_CANDIDATES_PATH, 'r', encoding='utf-8') as f:
            filtered_candidates = json.load(f)
    else:
        raise ValueError("İşlenecek aday bulunamadı! Lütfen 5. adımı çalıştırın.")

words_to_parse = [w for w in filtered_candidates.keys() if w not in parsed_results]
print(f"Çözümlenecek kalan kelime sayısı: {len(words_to_parse):,}")

SYSTEM_PROMPT = (
    "Sen Türkçe dilbilim ve morfoloji uzmanısın. Verilen Türkçe kelime listesini analiz et. "
    "Her kelime için kök (lemma), tür (pos) ve morfolojik nitelikleri (attributes) çıkar.\n"
    "Örnek Giriş: [\"kitaplar\", \"yapıverdi\", \"sözlüğümüzün\"]\n"
    "Örnek Çıktı:\n"
    "[\n"
    "  {\"word\": \"kitaplar\", \"lemma\": \"kitap\", \"pos\": \"Noun\", \"attributes\": [\"Voicing\"]},\n"
    "  {\"word\": \"yapıverdi\", \"lemma\": \"yapıvermek\", \"pos\": \"Verb\", \"attributes\": []},\n"
    "  {\"word\": \"sözlüğümüzün\", \"lemma\": \"sözlük\", \"pos\": \"Noun\", \"attributes\": [\"Voicing\"]}\n"
    "]\n"
    "Kurallar:\n"
    "- Fiiller mastar (-mak/-mek) halinde olmalı.\n"
    "- pos: 'Noun', 'Verb', 'Adjective' veya 'Adverb' olmalı.\n"
    "- attributes: 'Voicing' (kitap->kitabı, renk->rengi), 'LastVowelDrop' (burun->burnu), 'CompoundP3sg' (gökkuşağı, adabalığı).\n"
    "- Anlamsız yazım hatası veya yabancıysa null döndür.\n"
    "- SADECE JSON formatında bir liste döndür."
)

BATCH_SIZE = 10
for i in tqdm(range(0, len(words_to_parse), BATCH_SIZE), desc="Morfolojik Analiz"):
    batch = words_to_parse[i:i + BATCH_SIZE]
    user_prompt = f"Kelimeler: {json.dumps(batch, ensure_ascii=False)}\nJSON çıktısı:"
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
        
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    parsed_list = []
    try:
        match = re.search(r'\[.*\]', response, re.DOTALL)
        if match:
            parsed_list = json.loads(match.group(0))
    except Exception:
        # Hatalı/kısmi JSON durumunda tek tek nesneleri regex ile kurtar
        for obj_str in re.findall(r'\{[^{}]*"word"[^{}]*\}', response):
            try:
                parsed_list.append(json.loads(obj_str))
            except Exception:
                pass

    for item in parsed_list:
        w = item.get('word')
        if w and w in filtered_candidates:
            parsed_results[w] = {
                **filtered_candidates[w],
                "lemma": item.get('lemma'),
                "pos": item.get('pos'),
                "attributes": item.get('attributes', [])
            }
            
    # Her 50 kelimede bir Google Drive'a kaydet
    if (i // BATCH_SIZE) % 5 == 0:
        with open(CHECKPOINT_PARSED, 'w', encoding='utf-8') as f:
            json.dump(parsed_results, f, ensure_ascii=False, indent=2)
        if len(parsed_results) > 0 and len(parsed_results) % 50 == 0:
            sample_w = list(parsed_results.keys())[-1]
            sample_res = parsed_results[sample_w]
            print(f"\nİlerleme: {len(parsed_results)}/{len(words_to_parse)} | Örnek: {sample_w} -> {sample_res.get('lemma')} ({sample_res.get('pos')})")

# Son halini kaydet
with open(CHECKPOINT_PARSED, 'w', encoding='utf-8') as f:
    json.dump(parsed_results, f, ensure_ascii=False, indent=2)

print(f"\nAnaliz tamamlandı! Toplam {len(parsed_results):,} kelime başarıyla Drive'a kaydedildi: {CHECKPOINT_PARSED}")

## 🎯 8. Adım: Sonuçların İndirilmesi ve Yerel Projeye Aktarılması

Google Drive'ınızda `MyDrive/turkspell_mining/parsed_results_checkpoint.json` dosyası oluşmuştur. Bu dosyayı yerel bilgisayarınızda şu komutla işleyebilirsiniz:

```powershell
python tools/ingest_free_colab_candidates.py --input raw_data/parsed_results_checkpoint.json
```